# Technical Challenge — Sourcing Analysis with AI

**Objetivo:** Analisar um dataset de sourcing/recrutamento para identificar os melhores canais, sinais de avanço e quando insistir ou desistir de um candidato.

**Uso de IA:** Modelo preditivo de Regressão Logística para estimar probabilidade de contratação por candidato.

## 1. Importações e Configuração

In [ ]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
import warnings
warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 50)
print('Imports OK')

## 2. Carregamento e Tratamento dos Dados

In [ ]:
df = pd.read_csv('../data/mock_sourcing_dataset_clean.csv')
bool_cols = ['response_received','screening_pass','interview1_pass','test_taken','offer_sent','hired']
for c in bool_cols:
    df[c] = df[c].map({True:True, False:False, 'True':True, 'False':False})
num_cols = ['response_time_days','technical_test_score','behavior_score','manager_score','stage_duration_days','years_experience']
for c in num_cols:
    df[c] = pd.to_numeric(df[c], errors='coerce')
print(f'Shape: {df.shape}')
print(f'Hired total: {df["hired"].fillna(False).sum()}')

## 3. Análise de Funil

In [ ]:
funnel = pd.DataFrame({
    'stage': ['Sourced','Resposta','Screening','Entrevista 1','Teste','Oferta','Contratado'],
    'count': [
        len(df),
        df['response_received'].fillna(False).sum(),
        df['screening_pass'].fillna(False).sum(),
        df['interview1_pass'].fillna(False).sum(),
        df['test_taken'].fillna(False).sum(),
        df['offer_sent'].fillna(False).sum(),
        df['hired'].fillna(False).sum()
    ]
})
funnel['pct_total'] = (funnel['count'] / funnel['count'].iloc[0] * 100).round(1)
funnel['drop_pct'] = funnel['count'].pct_change().mul(100).round(1)
print(funnel.to_string(index=False))

## 4. Análise por Canal de Sourcing

In [ ]:
channel = df.groupby('source_channel').agg(
    candidatos=('candidate_id','count'),
    taxa_resposta=('response_received','mean'),
    taxa_screening=('screening_pass','mean'),
    taxa_entrevista=('interview1_pass','mean'),
    taxa_teste=('test_taken','mean'),
    taxa_oferta=('offer_sent','mean'),
    taxa_hire=('hired','mean'),
    media_resp_dias=('response_time_days','mean'),
    media_nota_tecnica=('technical_test_score','mean')
).reset_index().sort_values('taxa_hire', ascending=False)
pct_cols = ['taxa_resposta','taxa_screening','taxa_entrevista','taxa_teste','taxa_oferta','taxa_hire']
channel_display = channel.copy()
for c in pct_cols:
    channel_display[c] = (channel_display[c]*100).round(1).astype(str) + '%'
print(channel_display.to_string(index=False))

## 5. Sinais de Avanço — Scores e Responsividade

In [ ]:
resp_time = df.groupby(df['hired'].fillna(False))['response_time_days'].agg(['mean','median','count'])
resp_time.index = ['Não contratado','Contratado']
print('Tempo médio de resposta (dias) por resultado:')
print(resp_time)
print()
scores = df.groupby(df['hired'].fillna(False))[['technical_test_score','behavior_score','manager_score']].mean()
scores.index = ['Não contratado','Contratado']
print('Média de scores por resultado:')
print(scores)

## 6. Motivos de Rejeição

In [ ]:
rejection = df[df['rejection_reason'].notna()]['rejection_reason'].value_counts()
print('Motivos de rejeição mais comuns:')
print(rejection)

## 7. Análise por Recrutador

In [ ]:
recruiter = df.groupby('recruiter').agg(
    candidatos=('candidate_id','count'),
    taxa_hire=('hired','mean'),
    taxa_resposta=('response_received','mean'),
    taxa_screening=('screening_pass','mean'),
    media_resp_dias=('response_time_days','mean')
).reset_index().sort_values('taxa_hire', ascending=False)
for c in ['taxa_hire','taxa_resposta','taxa_screening']:
    recruiter[c] = (recruiter[c]*100).round(1).astype(str) + '%'
print(recruiter.to_string(index=False))

## 8. IA — Modelo Preditivo de Probabilidade de Hire

Modelo de Regressão Logística para apoio à priorização de pipeline.
**Objetivo:** ajudar recrutadores a focarem nos candidatos com maior probabilidade de conversão — não substituir julgamento humano.

In [ ]:
X = pd.get_dummies(df[['source_channel','department','work_mode','seniority','location']], drop_first=True)
X['response_time_days'] = df['response_time_days'].fillna(df['response_time_days'].median())
X['years_experience'] = df['years_experience'].fillna(df['years_experience'].median())
y = df['hired'].fillna(False).astype(int)
model = LogisticRegression(max_iter=1000, random_state=42)
model.fit(X, y)
auc = roc_auc_score(y, model.predict_proba(X)[:,1])
print(f'AUC do modelo: {auc:.3f}')
print('AUC > 0.65 é útil para priorização de pipeline')

In [ ]:
coef = pd.Series(model.coef_[0], index=X.columns).sort_values(key=lambda s: s.abs(), ascending=False)
print('Top 15 variáveis mais associadas ao resultado:')
print(coef.head(15).to_string())
print()
print('(+) aumenta probabilidade de hire / (-) reduz probabilidade de hire')

In [ ]:
df['hire_probability'] = model.predict_proba(X)[:,1]
df['priority'] = pd.cut(df['hire_probability'], bins=[0,0.05,0.12,1.0], labels=['Baixa','Média','Alta'])
priority_summary = df.groupby('priority', observed=True).agg(
    total=('candidate_id','count'),
    contratados=('hired', lambda x: x.fillna(False).sum()),
    taxa_hire_real=('hired', lambda x: x.fillna(False).mean())
).reset_index()
priority_summary['taxa_hire_real'] = (priority_summary['taxa_hire_real']*100).round(1).astype(str) + '%'
print('Distribuição de candidatos por faixa de prioridade:')
print(priority_summary.to_string(index=False))

## 9. Regras Práticas para Recrutadores

**Quando insistir:**
- Candidato respondeu + passou screening + scores consistentes
- Canal de alta conversão (GitHub, Inbound, Hunting)
- Motivo de stall é operacional (timing, agenda), não de fit

**Quando parar:**
- Sem resposta após sequência padrão de contatos
- Múltiplos sinais fracos: baixa responsividade + scores ruins + canal de baixa conversão
- Motivos estruturais confirmados (salary mismatch, outra oferta aceita)

## 10. Conclusão

Os três principais alavancas identificadas são:

1. **Mix de canais**: GitHub, Inbound e Hunting geram as melhores taxas de hire finais
2. **Velocidade de engajamento**: menor tempo de resposta está correlacionado com maior avanço no funil
3. **Priorização disciplinada**: usar IA para ranquear pipeline concentra esforço onde a conversão é mais provável